In [1]:
import pandas as pd
import scipy.stats as stats
import numpy as np
import ast

In [2]:
def compute_accuracy_local(df):
    list_Trustworthiness = df["Trustworthiness"].tolist()
    list_Continuity = df["Continuity"].tolist()

    results = []
    for i in range(len(list_Trustworthiness)):
        results.append(round(0.5*list_Trustworthiness[i] + 0.5*list_Continuity[i], 2))
    return results

In [3]:
def compute_accuracy_global(df):
    list_Shephard = df["Shephard Diagram Correlation"].tolist()

    results = []
    for i in range(len(list_Shephard)):
        results.append(round(0.5*(list_Shephard[i] + 1), 2))
    return results

In [4]:
def compute_perception(df):
    list_NeighborhoodHit = df["7-Neighborhood Hit"].tolist()
    list_DistanceConsistency = df["Distance consistency"].tolist()

    results = []
    for i in range(len(list_NeighborhoodHit)):
        results.append(round(0.5*list_NeighborhoodHit[i] + 0.5*list_DistanceConsistency[i], 2))
    return results

## Generate Entries for Topic Models

In [5]:
def generate_dataframe(corpus_file_layouts, corpus_file_model):
    df_layouts_all = pd.read_csv(corpus_file_layouts)
    
    # only keep the layouts with a topic model, i.e., TM is either lda, lsi, or nmf
    excluded_embeddings = ['bert', 'bow', 'tfidf']
    df_layouts = df_layouts_all[~df_layouts_all['TM'].isin(excluded_embeddings)]
    
    # detect the model, i.e., linear combined must be removed
    list_TM_short = [TM.replace('_linear_combined', '') for TM in df_layouts["TM"].tolist()]
    df_layouts["TM_short"] = list_TM_short
    
    # detect the number of topics
    list_n_topics = [name.split("n_topics_")[1].split("_")[0] for name in df_layouts["Experiment"].tolist()]
    df_layouts['n_topics'] = list_n_topics
    
    # compute the aggregated local accuracy
    df_layouts["accuracy_local"] = compute_accuracy_local(df_layouts)

    # compute the aggregated gloal accuracy
    df_layouts["accuracy_global"] = compute_accuracy_global(df_layouts)
    
    # compute the aggregated perception
    df_layouts["perception"] = compute_perception(df_layouts)
    
    # derive the coherence measures
    df_model = pd.read_csv(corpus_file_model)
    
    list_coherence_c_v = []
    list_coherence_u_mass = []
    list_coherence_c_uci = []
    list_coherence_c_npmi = []
    
    for i in range(df_layouts.shape[0]):
        model_type = list_TM_short[i]
        n_topics = list_n_topics[i]
        list_coherence_c_v.append(df_model[(df_model["model_type"] == model_type) & (df_model["n_topics"] == int(n_topics))]['coherence_c_v'].tolist()[0])
        list_coherence_u_mass.append(df_model[(df_model["model_type"] == model_type) & (df_model["n_topics"] == int(n_topics))]['coherence_u_mass'].tolist()[0])
        list_coherence_c_uci.append(df_model[(df_model["model_type"] == model_type) & (df_model["n_topics"] == int(n_topics))]['coherence_c_uci'].tolist()[0])
        list_coherence_c_npmi.append(df_model[(df_model["model_type"] == model_type) & (df_model["n_topics"] == int(n_topics))]['coherence_c_npmi'].tolist()[0])

    df_layouts['K'] = [int(n_topics) for n_topics in list_n_topics] 
    df_layouts['coherence_c_v'] = list_coherence_c_v
    df_layouts['coherence_u_mass'] = list_coherence_u_mass
    df_layouts['coherence_c_uci'] = list_coherence_c_uci
    df_layouts['coherence_c_npmi'] = list_coherence_c_npmi
    
    return df_layouts

In [6]:
print("Processing Emails")
df_emails = generate_dataframe(corpus_file_layouts = "results_cluster/cur_res/full_res_emails.csv", corpus_file_model = "model_evaluations/emails_model_evaluation.csv")
df_emails = df_emails[df_emails["K"] >= 8]

print("Processing 20 Newsgroups")
df_20newsgroups =  generate_dataframe(corpus_file_layouts = "results_cluster/cur_res/full_res_20_newsgroups.csv", corpus_file_model = "model_evaluations/20_newsgroups_model_evaluation.csv")
df_20newsgroups = df_20newsgroups[df_20newsgroups["K"] >= 20]

print("Processing BBC")
df_bbc = generate_dataframe(corpus_file_layouts = "results_cluster/cur_res/full_res_bbc_news.csv", corpus_file_model = "model_evaluations/bbc_news_model_evaluation.csv")
df_bbc = df_bbc[df_bbc["K"] >= 10]

print("Processing Lyrics")
df_lyrics = generate_dataframe(corpus_file_layouts = "results_cluster/cur_res/full_res_lyrics.csv", corpus_file_model = "model_evaluations/lyrics_model_evaluation.csv")
df_lyrics = df_lyrics[df_lyrics["K"] >= 8]

print("Processing Reuters")
df_reuters = generate_dataframe(corpus_file_layouts = "results_cluster/cur_res/full_res_reuters.csv", corpus_file_model = "model_evaluations/reuters_model_evaluation.csv")
df_reuters = df_reuters[df_reuters["K"] >= 10]

print("Processing Seven Categories")
df_7categories = generate_dataframe(corpus_file_layouts = "results_cluster/cur_res/full_res_seven_categories.csv", corpus_file_model = "model_evaluations/seven_categories_model_evaluation.csv")
df_7categories = df_7categories[df_7categories["K"] >= 14]

Processing Emails


C:\Users\Daniel Atzberger\AppData\Local\Temp\ipykernel_15464\3305303163.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_layouts["TM_short"] = list_TM_short
C:\Users\Daniel Atzberger\AppData\Local\Temp\ipykernel_15464\3305303163.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_layouts['n_topics'] = list_n_topics
C:\Users\Daniel Atzberger\AppData\Local\Temp\ipykernel_15464\3305303163.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .lo

Processing 20 Newsgroups


C:\Users\Daniel Atzberger\AppData\Local\Temp\ipykernel_15464\3305303163.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_layouts['K'] = [int(n_topics) for n_topics in list_n_topics]
C:\Users\Daniel Atzberger\AppData\Local\Temp\ipykernel_15464\3305303163.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_layouts['coherence_c_v'] = list_coherence_c_v
C:\Users\Daniel Atzberger\AppData\Local\Temp\ipykernel_15464\3305303163.py:43: SettingWithCopyWarning: 
A value is trying to be set on a copy of a sl

Processing BBC


...

C:\Users\Daniel Atzberger\AppData\Local\Temp\ipykernel_15464\3305303163.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_layouts['K'] = [int(n_topics) for n_topics in list_n_topics]
C:\Users\Daniel Atzberger\AppData\Local\Temp\ipykernel_15464\3305303163.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_layouts['coherence_c_v'] = list_coherence_c_v
C:\Users\Daniel Atzberger\AppData\Local\Temp\ipykernel_15464\3305303163.py:43: SettingWithCopyWarning: 
A value is trying to be set on a copy of a sl

Processing Reuters


C:\Users\Daniel Atzberger\AppData\Local\Temp\ipykernel_15464\3305303163.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_layouts['K'] = [int(n_topics) for n_topics in list_n_topics]
C:\Users\Daniel Atzberger\AppData\Local\Temp\ipykernel_15464\3305303163.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_layouts['coherence_c_v'] = list_coherence_c_v
C:\Users\Daniel Atzberger\AppData\Local\Temp\ipykernel_15464\3305303163.py:43: SettingWithCopyWarning: 
A value is trying to be set on a copy of a sl

Processing Seven Categories


C:\Users\Daniel Atzberger\AppData\Local\Temp\ipykernel_15464\3305303163.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_layouts['K'] = [int(n_topics) for n_topics in list_n_topics]
C:\Users\Daniel Atzberger\AppData\Local\Temp\ipykernel_15464\3305303163.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_layouts['coherence_c_v'] = list_coherence_c_v
C:\Users\Daniel Atzberger\AppData\Local\Temp\ipykernel_15464\3305303163.py:43: SettingWithCopyWarning: 
A value is trying to be set on a copy of a sl

In [7]:
df_emails.head(8)

,Experiment,Trustworthiness,Continuity,Shephard Diagram Correlation,Normalized Stress,7-Neighborhood Hit,Calinski-Harabasz-Index,Silhouette coefficient,Davies-Bouldin-Index,SDBW validity index,...,TM_short,n_topics,accuracy_local,accuracy_global,perception,K,coherence_c_v,coherence_u_mass,coherence_c_uci,coherence_c_npmi
1135,emails_lda_linear_combined_n_topics_12_alpha_a...,0.528130,0.572542,-0.308098,2.229384,0.489863,0.520001,-0.065064,701.574240,1.510636,...,lda,12,0.55,0.35,0.39,12,0.362987,-1.716848,-0.938175,-0.030855
1136,emails_lda_linear_combined_n_topics_12_alpha_a...,0.528130,0.572542,-0.308098,2.229384,0.489863,0.520001,-0.065064,701.574240,1.510636,...,lda,12,0.55,0.35,0.39,12,0.362987,-1.716848,-0.938175,-0.030855
1137,emails_lda_linear_combined_n_topics_12_alpha_a...,0.528130,0.572542,-0.308098,2.229384,0.489863,0.520001,-0.065064,701.574240,1.510636,...,lda,12,0.55,0.35,0.39,12,0.362987,-1.716848,-0.938175,-0.030855
1138,emails_lda_linear_combined_n_topics_12_alpha_a...,0.528130,0.572542,-0.308098,2.229384,0.489863,0.520001,-0.065064,701.574240,1.510636,...,lda,12,0.55,0.35,0.39,12,0.362987,-1.716848,-0.938175,-0.030855
1139,emails_lda_linear_combined_n_topics_12_alpha_a...,0.528130,0.572542,-0.308098,2.229384,0.489863,0.520001,-0.065064,701.574240,1.510636,...,lda,12,0.55,0.35,0.39,12,0.362987,-1.716848,-0.938175,-0.030855
1140,emails_lda_linear_combined_n_topics_12_alpha_a...,0.993230,0.957140,0.521104,6504.845942,0.497342,326.421570,-0.026970,16.920979,1.513771,...,lda,12,0.98,0.76,0.47,12,0.362987,-1.716848,-0.938175,-0.030855
1141,emails_lda_linear_combined_n_topics_12_alpha_a...,0.649691,0.671592,0.220328,1875.295085,0.421139,339.928141,-0.016469,22.749713,0.923680,...,lda,12,0.66,0.61,0.42,12,0.362987,-1.716848,-0.938175,-0.030855
1142,emails_lda_linear_combined_n_topics_12_alpha_a...,0.993102,0.955445,0.518905,5276.522173,0.491651,128.966761,-0.034470,29.418711,0.753860,...,lda,12,0.97,0.76,0.38,12,0.362987,-1.716848,-0.938175,-0.030855


In [8]:
set(df_7categories["K"].tolist())

{14, 21, 28, 35}

In [9]:
list_df = [df_emails, df_20newsgroups, df_bbc, df_lyrics, df_reuters, df_7categories]
df = pd.concat(list_df, axis=0, ignore_index=True)
df.shape

(51552, 24)

In [10]:
columns = ['embedding', 'DR', 'layout_quality_measure', 'embedding_quality_measure', 'value']
df_result = pd.DataFrame(columns = columns)

TM_list = ['lda', 'lsi', 'lsi_tfidf', 'nmf', 'nmf_tfidf']
DR_list = ['mds', 'som', 'tsne', 'umap']

for TM in TM_list:
    for DR in DR_list:
        df_selected = df[(df["TM_short"] == TM) & (df["DR"] == DR)]
        list_accuracy_local = df_selected["accuracy_local"].tolist()
        list_accuracy_global = df_selected["accuracy_global"].tolist()
        list_perception = df_selected["perception"].tolist()
        
        list_coherence_c_v = df_selected["coherence_c_v"].tolist()
        value_acc_loc = stats.kendalltau(list_accuracy_local, list_coherence_c_v)[0]
        new_row_acc_loc_c_v = {'embedding': TM, 'DR': DR, 'layout_quality_measure': 'acc_loc', 'embedding_quality_measure': 'c_v', 'value':value_acc_loc}
        df_result = pd.concat([df_result, pd.DataFrame([new_row_acc_loc_c_v])], ignore_index=True)

        value_acc_glo = stats.kendalltau(list_accuracy_global, list_coherence_c_v)[0]
        new_row_acc_glo_c_v = {'embedding': TM, 'DR': DR, 'layout_quality_measure': 'acc_glo', 'embedding_quality_measure': 'c_v', 'value':value_acc_glo}
        df_result = pd.concat([df_result, pd.DataFrame([new_row_acc_glo_c_v])], ignore_index=True)
        
        value_per = stats.kendalltau(list_perception, list_coherence_c_v)[0]
        new_row_per_c_v = {'embedding': TM, 'DR': DR, 'layout_quality_measure': 'per', 'embedding_quality_measure': 'c_v', 'value':value_per}
        #df_result.append(new_row_per_c_v, ignore_index=True)
        df_result = pd.concat([df_result, pd.DataFrame([new_row_per_c_v])], ignore_index=True)

        
        list_coherence_u_mass = df_selected["coherence_u_mass"].tolist()
        value_acc_loc = stats.kendalltau(list_accuracy_local, list_coherence_u_mass)[0]
        new_row_acc_loc_u_mass = {'embedding': TM, 'DR': DR, 'layout_quality_measure': 'acc_loc', 'embedding_quality_measure': 'u_mass', 'value':value_acc_loc}
        df_result = pd.concat([df_result, pd.DataFrame([new_row_acc_loc_u_mass])], ignore_index=True)

        value_acc_glo = stats.kendalltau(list_accuracy_global, list_coherence_u_mass)[0]
        new_row_acc_glo_u_mass = {'embedding': TM, 'DR': DR, 'layout_quality_measure': 'acc_glo', 'embedding_quality_measure': 'u_mass', 'value':value_acc_glo}
        df_result = pd.concat([df_result, pd.DataFrame([new_row_acc_glo_u_mass])], ignore_index=True)
        
        value_per = stats.kendalltau(list_perception, list_coherence_u_mass)[0]
        new_row_per_u_mass = {'embedding': TM, 'DR': DR, 'layout_quality_measure': 'per', 'embedding_quality_measure': 'u_mass', 'value':value_per}
        #df_result.append(new_row_per_u_mass, ignore_index=True)
        df_result = pd.concat([df_result, pd.DataFrame([new_row_per_u_mass])], ignore_index=True)

        
        list_coherence_c_uci = df_selected["coherence_c_uci"].tolist()
        value_acc_loc = stats.kendalltau(list_accuracy_local, list_coherence_c_uci)[0]
        new_row_acc_loc_c_uci = {'embedding': TM, 'DR': DR, 'layout_quality_measure': 'acc_loc', 'embedding_quality_measure': 'c_uci', 'value':value_acc_loc}
        df_result = pd.concat([df_result, pd.DataFrame([new_row_acc_loc_c_uci])], ignore_index=True)

        value_acc_glo = stats.kendalltau(list_accuracy_global, list_coherence_c_uci)[0]
        new_row_acc_glo_c_uci = {'embedding': TM, 'DR': DR, 'layout_quality_measure': 'acc_glo', 'embedding_quality_measure': 'c_uci', 'value':value_acc_glo}
        df_result = pd.concat([df_result, pd.DataFrame([new_row_acc_glo_c_uci])], ignore_index=True)
        
        value_per = stats.kendalltau(list_perception, list_coherence_c_uci)[0]
        new_row_per_c_uci = {'embedding': TM, 'DR': DR, 'layout_quality_measure': 'per', 'embedding_quality_measure': 'c_uci', 'value':value_per}
        #df_result.append(new_row_per_u_uci, ignore_index=True)
        df_result = pd.concat([df_result, pd.DataFrame([new_row_per_c_uci])], ignore_index=True)

        
        list_coherence_c_npmi = df_selected["coherence_c_npmi"].tolist()
        value_acc_loc = stats.kendalltau(list_accuracy_local, list_coherence_c_npmi)[0]
        new_row_acc_loc_c_npmi = {'embedding': TM, 'DR': DR, 'layout_quality_measure': 'acc_loc', 'embedding_quality_measure': 'c_npmi', 'value':value_acc_loc}
        df_result = pd.concat([df_result, pd.DataFrame([new_row_acc_loc_c_npmi])], ignore_index=True)

        value_acc_glo = stats.kendalltau(list_accuracy_global, list_coherence_c_npmi)[0]
        new_row_acc_glo_c_npmi = {'embedding': TM, 'DR': DR, 'layout_quality_measure': 'acc_glo', 'embedding_quality_measure': 'c_npmi', 'value':value_acc_glo}
        df_result = pd.concat([df_result, pd.DataFrame([new_row_acc_glo_c_npmi])], ignore_index=True)
        
        value_per = stats.kendalltau(list_perception, list_coherence_c_npmi)[0]
        new_row_per_c_npmi = {'embedding': TM, 'DR': DR, 'layout_quality_measure': 'per', 'embedding_quality_measure': 'c_npmi', 'value':value_per}
        #df_result.append(new_row_per_c_npmi, ignore_index=True)
        df_result = pd.concat([df_result, pd.DataFrame([new_row_per_c_npmi])], ignore_index=True)

C:\Users\Daniel Atzberger\AppData\Local\Temp\ipykernel_15464\4004906952.py:17: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_result = pd.concat([df_result, pd.DataFrame([new_row_acc_loc_c_v])], ignore_index=True)


In [11]:
df_result

,embedding,DR,layout_quality_measure,embedding_quality_measure,value
0,lda,mds,acc_loc,c_v,-0.173093
1,lda,mds,acc_glo,c_v,0.192826
2,lda,mds,per,c_v,-0.325689
3,lda,mds,acc_loc,u_mass,0.011256
4,lda,mds,acc_glo,u_mass,-0.180286
...,...,...,...,...,...
235,nmf_tfidf,umap,acc_glo,c_uci,0.203711
236,nmf_tfidf,umap,per,c_uci,-0.416733
237,nmf_tfidf,umap,acc_loc,c_npmi,0.068800
238,nmf_tfidf,umap,acc_glo,c_npmi,0.211838


## Bert Models

In [12]:
# https://www.sbert.net/docs/sentence_transformer/pretrained_models.html
df_quality_bert = pd.DataFrame({'model': ['bert_all-MiniLM-L6-v2', 'bert_all-distilroberta-v1', 'bert_all-mpnet-base-v2', 'bert_paraphrase-MiniLM-L3-v2', 'bert_paraphrase-albert-small-v2'],
                               'performance_sentence_embeddings': [68.06, 68.06, 69.57, 62.29, 64.46],
                              'performance_semantic_search': [49.54, 50.94, 57.02, 39.19, 40.04]})
df_quality_bert.head(5)

,model,performance_sentence_embeddings,performance_semantic_search
0,bert_all-MiniLM-L6-v2,68.06,49.54
1,bert_all-distilroberta-v1,68.06,50.94
2,bert_all-mpnet-base-v2,69.57,57.02
3,bert_paraphrase-MiniLM-L3-v2,62.29,39.19
4,bert_paraphrase-albert-small-v2,64.46,40.04


In [13]:
def get_model(experiment):
    model_list = ['bert_all-MiniLM-L6-v2', 'bert_all-distilroberta-v1', 'bert_all-mpnet-base-v2', 'bert_paraphrase-MiniLM-L3-v2', 'bert_paraphrase-albert-small-v2']
    for model in model_list:
        if model in experiment:
            return model

In [14]:
def get_model_performance_sentence_embeddings(model):
    return df_quality_bert[df_quality_bert["model"] == model]["performance_sentence_embeddings"].tolist()[0]

In [15]:
def get_model_performance_semantic_search(model):
    return df_quality_bert[df_quality_bert["model"] == model]["performance_semantic_search"].tolist()[0]

In [16]:
def generate_dataframe_bert(corpus_file_layouts):
    df_layouts_all = pd.read_csv(corpus_file_layouts)
    
    # only keep the layouts with BERT
    df_layouts = df_layouts_all[df_layouts_all['TM'].isin(['bert'])]
    
    # detect the model
    experiments_list = df_layouts["Experiment"].tolist()
    model_list = [get_model(experiment) for experiment in experiments_list]
    performance_sentence_embedding_list = [get_model_performance_sentence_embeddings(model) for model in model_list]
    performance_semantic_search_list = [get_model_performance_semantic_search(model) for model in model_list]

    df_layouts["performance_sentence_embedding"] = performance_sentence_embedding_list
    df_layouts["performance_semantic_search"] = performance_semantic_search_list
    
    # compute the aggregated local accuracy
    df_layouts["accuracy_local"] = compute_accuracy_local(df_layouts)

    # compute the aggregated global accuracy
    df_layouts["accuracy_global"] = compute_accuracy_global(df_layouts)
    
    # compute the aggregated perception
    df_layouts["perception"] = compute_perception(df_layouts)
    
    return df_layouts

In [17]:
df_emails_bert = generate_dataframe_bert(corpus_file_layouts = "results_cluster/cur_res/full_res_emails.csv")
df_20newsgroups_bert =  generate_dataframe_bert(corpus_file_layouts = "results_cluster/cur_res/full_res_20_newsgroups.csv")
df_bbc_bert = generate_dataframe_bert(corpus_file_layouts = "results_cluster/cur_res/full_res_bbc_news.csv")
df_lyrics_bert = generate_dataframe_bert(corpus_file_layouts = "results_cluster/cur_res/full_res_lyrics.csv")
df_reuters_bert = generate_dataframe_bert(corpus_file_layouts = "results_cluster/cur_res/full_res_reuters.csv")
df_7categories_bert = generate_dataframe_bert(corpus_file_layouts = "results_cluster/cur_res/full_res_seven_categories.csv")

C:\Users\Daniel Atzberger\AppData\Local\Temp\ipykernel_15464\773101083.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_layouts["performance_sentence_embedding"] = performance_sentence_embedding_list
C:\Users\Daniel Atzberger\AppData\Local\Temp\ipykernel_15464\773101083.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_layouts["performance_semantic_search"] = performance_semantic_search_list
C:\Users\Daniel Atzberger\AppData\Local\Temp\ipykernel_15464\773101083.py:17: SettingWithCopyWarning: 
A

In [18]:
list_df_bert = [df_emails_bert, df_20newsgroups_bert, df_bbc_bert, df_lyrics_bert, df_reuters_bert, df_7categories_bert]
df_bert = pd.concat(list_df_bert, axis=0, ignore_index=True)
df_bert.shape

(6749, 19)

In [19]:
df_bert.head()

,Experiment,Trustworthiness,Continuity,Shephard Diagram Correlation,Normalized Stress,7-Neighborhood Hit,Calinski-Harabasz-Index,Silhouette coefficient,Davies-Bouldin-Index,SDBW validity index,Distance consistency,Complete List of Hyperparameters,DR,TM,performance_sentence_embedding,performance_semantic_search,accuracy_local,accuracy_global,perception
0,emails_bert_all-MiniLM-L6-v2_n_categories_4_md...,0.537341,0.511359,-0.354358,0.409996,0.446948,0.875768,-0.059207,122.718519,1.642598,0.216661,"{'mds': {'max_iter': 200, 'dissimilarity_metri...",mds,bert,68.06,49.54,0.52,0.32,0.33
1,emails_bert_all-MiniLM-L6-v2_n_categories_4_md...,0.537341,0.511359,-0.354358,0.409996,0.446948,0.875768,-0.059207,122.718519,1.642598,0.216661,"{'mds': {'max_iter': 300, 'dissimilarity_metri...",mds,bert,68.06,49.54,0.52,0.32,0.33
2,emails_bert_all-MiniLM-L6-v2_n_categories_4_md...,0.537341,0.511359,-0.354358,0.409996,0.446948,0.875768,-0.059207,122.718519,1.642598,0.216661,"{'mds': {'max_iter': 150, 'dissimilarity_metri...",mds,bert,68.06,49.54,0.52,0.32,0.33
3,emails_bert_all-MiniLM-L6-v2_n_categories_4_md...,0.539918,0.511557,-0.354343,0.409996,0.444220,0.872840,-0.059202,124.491845,1.663883,0.216112,"{'mds': {'max_iter': 100, 'dissimilarity_metri...",mds,bert,68.06,49.54,0.53,0.32,0.33
4,emails_bert_all-MiniLM-L6-v2_n_categories_4_md...,0.537341,0.511359,-0.354358,0.409996,0.446948,0.875768,-0.059207,122.718519,1.642598,0.216661,"{'mds': {'max_iter': 250, 'dissimilarity_metri...",mds,bert,68.06,49.54,0.52,0.32,0.33


In [20]:
columns = ['embedding', 'DR', 'layout_quality_measure', 'embedding_quality_measure', 'value']
df_bert_result = pd.DataFrame(columns = columns)

TM_list = ['bert']
DR_list = ['mds', 'som', 'tsne', 'umap']

for TM in TM_list:
    for DR in DR_list:
        df_selected = df_bert[df_bert["DR"] == DR]
        list_accuracy_local = df_selected["accuracy_local"].tolist()
        list_accuracy_global = df_selected["accuracy_global"].tolist()
        list_perception = df_selected["perception"].tolist()
        
        list_performance_sentence_embeddings = df_selected["performance_sentence_embedding"].tolist()
        value_acc_loc = stats.kendalltau(list_accuracy_local, list_performance_sentence_embeddings)[0]
        new_row_acc_loc_sentence_embeddings = {'embedding': TM, 'DR': DR, 'layout_quality_measure': 'acc_loc', 'embedding_quality_measure': 'performance_sentence_embeddings', 'value':value_acc_loc}
        df_bert_result = pd.concat([df_bert_result, pd.DataFrame([new_row_acc_loc_sentence_embeddings])], ignore_index=True)

        value_acc_glo = stats.kendalltau(list_accuracy_global, list_performance_sentence_embeddings)[0]
        new_row_acc_glo_sentence_embeddings = {'embedding': TM, 'DR': DR, 'layout_quality_measure': 'acc_glo', 'embedding_quality_measure': 'performance_sentence_embeddings', 'value':value_acc_glo}
        df_bert_result = pd.concat([df_bert_result, pd.DataFrame([new_row_acc_glo_sentence_embeddings])], ignore_index=True)
        
        value_per = stats.kendalltau(list_perception, list_performance_sentence_embeddings)[0]
        new_row_per_sentence_embeddings = {'embedding': TM, 'DR': DR, 'layout_quality_measure': 'per', 'embedding_quality_measure': 'performance_sentence_embeddings', 'value':value_per}
        #df_result.append(new_row_per_c_v, ignore_index=True)
        df_bert_result = pd.concat([df_bert_result, pd.DataFrame([new_row_per_sentence_embeddings])], ignore_index=True)


        list_performance_semantic_search = df_selected["performance_semantic_search"].tolist()
        value_acc_loc = stats.kendalltau(list_accuracy_local, list_performance_semantic_search)[0]
        new_row_acc_loc_semantic_search = {'embedding': TM, 'DR': DR, 'layout_quality_measure': 'acc_loc', 'embedding_quality_measure': 'performance_semantic_search', 'value':value_acc_loc}
        df_bert_result = pd.concat([df_bert_result, pd.DataFrame([new_row_acc_loc_semantic_search])], ignore_index=True)

        value_acc_glo = stats.kendalltau(list_accuracy_global, list_performance_semantic_search)[0]
        new_row_acc_glo_semantic_search = {'embedding': TM, 'DR': DR, 'layout_quality_measure': 'acc_glo', 'embedding_quality_measure': 'performance_semantic_search', 'value':value_acc_glo}
        df_bert_result = pd.concat([df_bert_result, pd.DataFrame([new_row_acc_glo_semantic_search])], ignore_index=True)
        
        value_per = stats.kendalltau(list_perception, list_performance_semantic_search)[0]
        new_row_per_semantic_search = {'embedding': TM, 'DR': DR, 'layout_quality_measure': 'per', 'embedding_quality_measure': 'performance_semantic_search', 'value':value_per}
        #df_result.append(new_row_per_c_v, ignore_index=True)
        df_bert_result = pd.concat([df_bert_result, pd.DataFrame([new_row_per_semantic_search])], ignore_index=True)
        

C:\Users\Daniel Atzberger\AppData\Local\Temp\ipykernel_15464\3741771899.py:17: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_bert_result = pd.concat([df_bert_result, pd.DataFrame([new_row_acc_loc_sentence_embeddings])], ignore_index=True)


In [21]:
df_bert_result

,embedding,DR,layout_quality_measure,embedding_quality_measure,value
0,bert,mds,acc_loc,performance_sentence_embeddings,0.185039
1,bert,mds,acc_glo,performance_sentence_embeddings,-0.468657
2,bert,mds,per,performance_sentence_embeddings,0.034934
3,bert,mds,acc_loc,performance_semantic_search,0.178561
4,bert,mds,acc_glo,performance_semantic_search,-0.473118
5,bert,mds,per,performance_semantic_search,0.008417
6,bert,som,acc_loc,performance_sentence_embeddings,0.181486
7,bert,som,acc_glo,performance_sentence_embeddings,0.266881
8,bert,som,per,performance_sentence_embeddings,0.079131
9,bert,som,acc_loc,performance_semantic_search,0.175968


In [22]:
df_total = pd.concat([df_result, df_bert_result], ignore_index = True)
df_total.shape

(264, 5)

In [23]:
df_total

,embedding,DR,layout_quality_measure,embedding_quality_measure,value
0,lda,mds,acc_loc,c_v,-0.173093
1,lda,mds,acc_glo,c_v,0.192826
2,lda,mds,per,c_v,-0.325689
3,lda,mds,acc_loc,u_mass,0.011256
4,lda,mds,acc_glo,u_mass,-0.180286
...,...,...,...,...,...
259,bert,umap,acc_glo,performance_sentence_embeddings,0.296318
260,bert,umap,per,performance_sentence_embeddings,0.131898
261,bert,umap,acc_loc,performance_semantic_search,0.211048
262,bert,umap,acc_glo,performance_semantic_search,0.310045


In [24]:
df_total["value"] = df_total["value"].round(2)
df_total

,embedding,DR,layout_quality_measure,embedding_quality_measure,value
0,lda,mds,acc_loc,c_v,-0.17
1,lda,mds,acc_glo,c_v,0.19
2,lda,mds,per,c_v,-0.33
3,lda,mds,acc_loc,u_mass,0.01
4,lda,mds,acc_glo,u_mass,-0.18
...,...,...,...,...,...
259,bert,umap,acc_glo,performance_sentence_embeddings,0.30
260,bert,umap,per,performance_sentence_embeddings,0.13
261,bert,umap,acc_loc,performance_semantic_search,0.21
262,bert,umap,acc_glo,performance_semantic_search,0.31


In [25]:
# df_total.to_csv("analysis-results/kendalls-tau.csv")
df_total.to_csv("analysis-results/kendalls-tau-ueberpruefung/kendalls-tau.csv")